# Tutorial 03: Benchmarking 5 Time-Series Models

This notebook runs a **side-by-side comparison** of all five models in the framework:

| Model | Type | Reference |
|---|---|---|
| LSTM | Recurrent | Hochreiter & Schmidhuber, 1997 |
| TCN | Convolutional | Bai et al., 2018 |
| Transformer | Attention | Vaswani et al., 2017 |
| PatchTST | Patching + Attention | Nie et al., ICLR 2023 |
| DLinear | Decomposition + Linear | Zeng et al., AAAI 2023 |

**Key takeaway:** DLinear frequently matches or beats complex Transformer models on
standard benchmarks — a well-known result in recent time-series literature.

**Dataset:** Synthetic 7-channel data (mimicking ETT structure). No download required.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

from models.lstm_forecasting import LSTMForecaster
from models.tcn_model import TCNForecaster
from models.transformer_ts import TimeSeriesTransformer
from models.patchtst import PatchTST
from models.dlinear import DLinear
from forecasting.pipeline import ForecastingPipeline
from evaluation.forecasting_metrics import print_metrics_comparison_table
from visualization.plot_utils import plot_model_comparison_bar, plot_benchmark_table, print_benchmark_table

## 1. Generate Dataset

We generate synthetic multivariate data with 7 channels (matching ETT structure: 6 load features + oil temperature).

In [ ]:
NUM_FEATURES = 7
SEQ_LEN  = 96
PRED_LEN = 96
N_TRAIN  = 4000
N_VAL    = 1000
BATCH    = 32
EPOCHS   = 8

rng = np.random.default_rng(99)
T = N_TRAIN + N_VAL + SEQ_LEN + PRED_LEN
t = np.linspace(0, 20 * np.pi, T)
raw = (np.sin(t[:, None] * rng.uniform(0.8, 1.2, NUM_FEATURES) + rng.uniform(0, np.pi, NUM_FEATURES))
       + 0.1 * rng.standard_normal((T, NUM_FEATURES))).astype(np.float32)

def make_loader(offset, n):
    xs = np.stack([raw[offset + i : offset + i + SEQ_LEN] for i in range(n)])
    ys = np.stack([raw[offset + i + SEQ_LEN : offset + i + SEQ_LEN + PRED_LEN] for i in range(n)])
    ds = TensorDataset(torch.tensor(xs), torch.tensor(ys))
    return DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)

train_loader = make_loader(0, N_TRAIN)
val_loader   = make_loader(N_TRAIN, N_VAL)

print(f"Dataset: {NUM_FEATURES} features, seq_len={SEQ_LEN}, pred_len={PRED_LEN}")
print(f"Train: {N_TRAIN} samples  |  Val: {N_VAL} samples")

## 2. Define Models

All models are configured with **comparable capacity** (~same number of trainable parameters).

In [ ]:
def make_models():
    return {
        "LSTM": LSTMForecaster(
            num_features=NUM_FEATURES, hidden_dim=64, num_layers=2,
            out_features=PRED_LEN * NUM_FEATURES,
        ),
        "TCN": TCNForecaster(
            num_features=NUM_FEATURES, num_channels=[32, 64, 64], kernel_size=3,
            out_features=PRED_LEN * NUM_FEATURES,
        ),
        "Transformer": TimeSeriesTransformer(
            num_features=NUM_FEATURES, d_model=64, nhead=4, num_layers=2,
            dim_feedforward=128, out_features=PRED_LEN * NUM_FEATURES,
        ),
        "PatchTST": PatchTST(
            num_features=NUM_FEATURES, seq_len=SEQ_LEN, pred_len=PRED_LEN,
            patch_len=16, stride=8, d_model=64, nhead=4, num_layers=2,
        ),
        "DLinear": DLinear(
            seq_len=SEQ_LEN, pred_len=PRED_LEN, num_features=NUM_FEATURES,
            kernel_size=25, individual=True,
        ),
    }

models = make_models()
for name, m in models.items():
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {name:<15}  {n:>8,} parameters")

## 3. Train All Models

In [ ]:
results = []

for model_name, model in models.items():
    print(f"\n{'─'*50}")
    print(f"  Training {model_name}")
    print(f"{'─'*50}")
    
    pipeline = ForecastingPipeline(
        model=model,
        learning_rate=1e-3,
        model_name=model_name,
        dataset_name="ETT_synthetic",
        pred_len=PRED_LEN,
    )
    metrics = pipeline.fit(train_loader, val_loader, epochs=EPOCHS)
    results.append(metrics)

print("\nAll models trained.")

## 4. Compare Results

In [ ]:
print("\n" + "="*60)
print("  MODEL COMPARISON — ETT Synthetic  (pred_len=96)")
print("="*60)
print_metrics_comparison_table(results)

In [ ]:
model_names = [m.model_name for m in results]
rmse_values = [m.rmse for m in results]
mae_values  = [m.mae for m in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, values, metric in [(axes[0], rmse_values, "RMSE"), (axes[1], mae_values, "MAE")]:
    colors = ['#2ecc71' if v == min(values) else '#3498db' for v in values]
    bars = ax.bar(model_names, values, color=colors, edgecolor='white')
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} — lower is better")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle("Model Comparison — ETT Synthetic (pred_len=96)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Print as a table — copy this into your README
rows = [m.to_dict() for m in results]
print_benchmark_table(rows, columns=["model", "RMSE", "MAE", "MAPE", "SMAPE"])

## 5. Key Observations

- **DLinear** is competitive with or better than heavier Transformer models despite having far fewer parameters — consistent with the findings in Zeng et al. (AAAI 2023).
- **PatchTST** typically outperforms the vanilla Transformer due to patching and channel-independence.
- **TCN** offers the best inference throughput due to full parallelism (no recurrence).
- **LSTM** remains a strong baseline for shorter sequences with clear temporal dependencies.

For production ICS deployments, consider the trade-off between model accuracy and inference latency on edge hardware.